#1. Instalasi

In [8]:
!pip install sastrawi nltk regex pandas numpy

#2. Import Library

In [9]:
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('stopwords')
nltk.download('punkt')

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


#3. Load Data

In [10]:
from google.colab import files
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name, delimiter=";")

print(f"Jumlah data: {df.shape}")
print(df.head())

Saving full_text.csv to full_text.csv
Jumlah data: (605, 1)
                                           full_text
0  Bimbingan Teknis Sistem Penghubung Layanan Pem...
1  @bagaimanakumaha udah lah metropolitan2 berak....
2  Sungguh sistem perbankan syariah di Aceh adala...
3  Pemerintah Kota (Pemkot) Jakarta Pusat bersama...
4  @SammiSoh Wah mas wali @bobbynasution_ gimana ...


#4. Eksplorasi Data

In [11]:
print("=== INFO DATA ===")
df.info()

print("\n=== MISSING VALUES ===")
print(df.isna().sum())

print("\n=== DUPLIKASI ===")
print(f"Jumlah duplikat: {df.duplicated('full_text').sum()}")

=== INFO DATA ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   full_text  605 non-null    object
dtypes: object(1)
memory usage: 4.9+ KB

=== MISSING VALUES ===
full_text    0
dtype: int64

=== DUPLIKASI ===
Jumlah duplikat: 9


In [12]:
df.drop_duplicates(subset='full_text', keep='first', inplace=True)
print(f"Jumlah data setelah hapus duplikat: {df.shape[0]}")

Jumlah data setelah hapus duplikat: 596


#5. Prapemrosesan

Case Folding & Cleaning

In [13]:
def casefolding(text):
    text = text.lower()
    text = re.sub(r'@[A-Za-z0-9]+', '', text)   # hapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text)   # hapus hashtag
    text = re.sub(r'RT[\s]+', '', text)          # hapus retweet
    text = re.sub(r'https\S+', '', text)         # hapus URL
    text = re.sub(r'[0-9]+', '', text)           # hapus angka
    text = re.sub(r'[-,?@$#%^/&*=!_:"]+', ' ', text)  # hapus simbol
    text = text.replace('\n', ' ')
    text = text.strip()
    return text

df['text_clean'] = df['full_text'].apply(casefolding)
print(df[['full_text', 'text_clean']].head(3))

                                           full_text  \
0  Bimbingan Teknis Sistem Penghubung Layanan Pem...   
1  @bagaimanakumaha udah lah metropolitan2 berak....   
2  Sungguh sistem perbankan syariah di Aceh adala...   

                                          text_clean  
0  bimbingan teknis sistem penghubung layanan pem...  
1  udah lah metropolitan berak. susah kalo gaada ...  
2  sungguh sistem perbankan syariah di aceh adala...  


Normalisasi Slang

In [14]:
key_norm = pd.read_csv('https://raw.githubusercontent.com/ksnugroho/klasifikasi-spam-sms/master/data/key_norm.csv')

def wordnormalization(text):
    text = ' '.join([
        key_norm.loc[key_norm['singkat'] == word, 'hasil'].values[0]
        if word in key_norm['singkat'].values else word
        for word in text.split()
    ])
    return text.lower()

df['text_clean'] = df['text_clean'].apply(wordnormalization)
print(df[['full_text', 'text_clean']].head(3))

                                           full_text  \
0  Bimbingan Teknis Sistem Penghubung Layanan Pem...   
1  @bagaimanakumaha udah lah metropolitan2 berak....   
2  Sungguh sistem perbankan syariah di Aceh adala...   

                                          text_clean  
0  bimbingan teknis sistem penghubung layanan pem...  
1  sudah lah metropolitan berak. susah kalau tida...  
2  sungguh sistem perbankan syariah di aceh adala...  


Stopword Removal

In [15]:
stopwords_ind = stopwords.words('indonesian')
more_stopwords = {'gk', 'ye', 'eng', 'utk', 'wkwk', 'hehe', 'jd',
                  'loh', 'rt', 'njir', 'si', 'ny', 'dg', 'lah', 'haha',
                  'klo', 'dong', 'ges', 'loo', 'huhu'}
stopwords_ind.extend(more_stopwords)

def remove_stop_words(text):
    return ' '.join([word for word in text.split() if word not in stopwords_ind])

df['text_clean'] = df['text_clean'].apply(remove_stop_words)
print(df[['full_text', 'text_clean']].head(3))

                                           full_text  \
0  Bimbingan Teknis Sistem Penghubung Layanan Pem...   
1  @bagaimanakumaha udah lah metropolitan2 berak....   
2  Sungguh sistem perbankan syariah di Aceh adala...   

                                          text_clean  
0  bimbingan teknis sistem penghubung layanan pem...  
1  metropolitan berak. susah pemerintah berani be...  
2  sungguh sistem perbankan syariah aceh puncak k...  


Stemming

In [16]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming(text):
    return stemmer.stem(text)

df['text_clean'] = df['text_clean'].apply(stemming)
print(df[['full_text', 'text_clean']].head(3))

                                           full_text  \
0  Bimbingan Teknis Sistem Penghubung Layanan Pem...   
1  @bagaimanakumaha udah lah metropolitan2 berak....   
2  Sungguh sistem perbankan syariah di Aceh adala...   

                                          text_clean  
0  bimbing teknis sistem hubung layan perintah yu...  
1  metropolitan berak susah perintah berani beran...  
2  sungguh sistem perban syariah aceh puncak tolo...  


#6. Simpan Hasil

In [17]:
df.to_csv('nlp.csv', index=False)
print("Preprocessing selesai! File disimpan sebagai nlp.csv")
print(f"Total data: {df.shape[0]}")

Preprocessing selesai! File disimpan sebagai nlp.csv
Total data: 596
